# Week 8, notebook 1: Discretization and boundary conditions

**HWRS 564a · Fall 2026**

The Week 7 model was a 100 m square with ten cells on a side. This week's model
is a piece of the Tucson basin, 15 km by 10 km. We use it to answer two
questions:

- How small do the cells need to be?
- How do you tell MODFLOW which part of a rectangular grid is aquifer?

## How to use this notebook

Run each cell with **Shift+Enter**. Cells marked `# YOUR TURN` have something
for you to write. Cells marked `# CHECK` test your answer and raise an error if
it is wrong.

Before you finish, run *Kernel → Restart Kernel and Run All Cells* to make sure
the notebook works from top to bottom.

## Learning objectives

By the end of this notebook you can:

1. Run the same problem at several cell sizes and say whether the answer has
   stopped changing
2. Explain why the head in a pumping cell is not the water level in the well
3. Use the three values of `ibound` to set active, inactive, and constant-head
   cells
4. Find active cells that are cut off from the rest of the model, and remove them

---

## Part 1: Setup

In [ ]:
from pathlib import Path

import flopy
import matplotlib.pyplot as plt
import numpy as np
from scipy import ndimage
from scipy.interpolate import RegularGridInterpolator

ROOT = Path("../..")  # course repo root
DATA = ROOT / "data"
MF_EXE = ROOT / "modflow" / "mf2005"
assert MF_EXE.exists(), (
    f"MODFLOW binary not found at {MF_EXE}. Run ./postbuild.sh from a terminal."
)

RUN = ROOT / "_run"

print("ready")

---

## Part 2: The model area

The model covers the north side of the Tucson basin. The file
`data/tucson_srtm_250m.csv` holds land-surface elevation from the Shuttle Radar
Topography Mission (SRTM), averaged onto a grid of 250 m cells: 40 rows and 60
columns. Row 0 is the north edge and column 0 is the west edge.

In [ ]:
NROW, NCOL, DELR = 40, 60, 250.0
land_surface = np.loadtxt(DATA / "tucson_srtm_250m.csv", delimiter=",")
print(f"shape {land_surface.shape}, "
      f"elevation {land_surface.min():.0f} to {land_surface.max():.0f} m")

EXTENT = [0, NCOL * DELR / 1000, 0, NROW * DELR / 1000]   # km, for plotting

fig, ax = plt.subplots(figsize=(8, 4.6))
im = ax.imshow(land_surface, cmap="terrain", extent=EXTENT)
fig.colorbar(im, ax=ax, label="land surface (m)", shrink=0.85)
ax.set_xlabel("distance east (km)")
ax.set_ylabel("distance north (km)")
ax.set_title("SRTM land surface on the model grid")
plt.tight_layout()
plt.show()

The low ground in the south-west is basin fill along the Rillito and Santa Cruz
rivers. The high ground in the north-east is the Santa Catalina Mountains, which
are bedrock. We come back to that in Part 4. For Part 3 we ignore the mountains
and treat the whole rectangle as aquifer.

---

## Part 3: Does the answer depend on the cell size?

Here is the model for this part:

- one confined layer, top at 600 m and bottom at 400 m
- hydraulic conductivity 12 m/d
- constant head of 620 m on the west edge and 700 m on the east edge
- recharge of 1.2e-4 m/d everywhere
- one well pumping 20,000 m³/d, 7,500 m east and 5,000 m south of the
  north-west corner

The function below builds and runs that model for any cell size. The domain
stays 15 km by 10 km, so smaller cells mean more of them. The well stays at the
same place on the ground, which means its row and column change with the cell
size.

We record two heads from each run: the head in the cell that holds the well,
and the head at a point 1 km west of the well. The second is interpolated
between cell centres so that it refers to the same point on every grid.

In [ ]:
def confined_box(cell_size, well_q=-20000.0):
    """Build and run the rectangular model. Returns (model, head, (row, col))."""
    ncol = int(15000 / cell_size)
    nrow = int(10000 / cell_size)
    ws = RUN / f"week08_box_{int(cell_size)}"
    ws.mkdir(parents=True, exist_ok=True)

    mf = flopy.modflow.Modflow("box", model_ws=str(ws), exe_name=str(MF_EXE))
    flopy.modflow.ModflowDis(mf, 1, nrow, ncol, delr=cell_size, delc=cell_size,
                             top=600.0, botm=400.0, nper=1, steady=True)

    ibound = np.ones((1, nrow, ncol), dtype=int)
    ibound[:, :, 0] = -1
    ibound[:, :, -1] = -1
    strt = np.full((1, nrow, ncol), 660.0)
    strt[:, :, 0] = 620.0
    strt[:, :, -1] = 700.0
    flopy.modflow.ModflowBas(mf, ibound=ibound, strt=strt)

    flopy.modflow.ModflowLpf(mf, hk=12.0, laytyp=0, ipakcb=53)

    # same location on the ground at every cell size
    wr, wc = int(5000 / cell_size), int(7500 / cell_size)
    flopy.modflow.ModflowWel(mf, stress_period_data={0: [[0, wr, wc, well_q]]},
                             ipakcb=53)
    flopy.modflow.ModflowRch(mf, rech=1.2e-4, ipakcb=53)
    flopy.modflow.ModflowPcg(mf)
    flopy.modflow.ModflowOc(mf, stress_period_data={(0, 0): ["save head", "save budget"]})

    mf.write_input()
    ok, buff = mf.run_model(silent=True, report=True)
    assert ok, f"{cell_size} m grid failed:\n" + "\n".join(buff[-15:])
    head = flopy.utils.HeadFile(str(ws / "box.hds")).get_data()
    return mf, head, (wr, wc)


def head_at(head, cell_size, x_east, y_south):
    """Head at a point, interpolated between cell centres. Distances in metres
    from the north-west corner."""
    nrow, ncol = head[0].shape
    y = (np.arange(nrow) + 0.5) * cell_size
    x = (np.arange(ncol) + 0.5) * cell_size
    return float(RegularGridInterpolator((y, x), head[0])([y_south, x_east])[0])


results = {}
for size in [500.0, 250.0, 125.0]:
    mf, head, (wr, wc) = confined_box(size)
    results[size] = {
        "n_cells": head[0].size,
        "well_head": float(head[0, wr, wc]),
        "near_head": head_at(head, size, 6500.0, 5000.0),   # 1 km west of the well
    }
    print(f"{size:6.0f} m  {results[size]['n_cells']:7,d} cells  "
          f"well cell {results[size]['well_head']:8.3f} m  "
          f"1 km west of the well {results[size]['near_head']:8.3f} m")

### YOUR TURN 1

Work out how much each head changes when the cell size is halved.

- `well_change_1`: well-cell head at 250 m minus well-cell head at 500 m
- `well_change_2`: well-cell head at 125 m minus well-cell head at 250 m
- `near_change_1` and `near_change_2`: the same two differences for the head
  1 km west of the well

In [ ]:
# YOUR TURN
well_change_1 = ...
well_change_2 = ...
near_change_1 = ...
near_change_2 = ...

In [ ]:
# CHECK
r = results
assert abs(well_change_1 - (r[250.0]["well_head"] - r[500.0]["well_head"])) < 1e-6, f"got {well_change_1}"
assert abs(well_change_2 - (r[125.0]["well_head"] - r[250.0]["well_head"])) < 1e-6, f"got {well_change_2}"
assert abs(near_change_1 - (r[250.0]["near_head"] - r[500.0]["near_head"])) < 1e-6, f"got {near_change_1}"
assert abs(near_change_2 - (r[125.0]["near_head"] - r[250.0]["near_head"])) < 1e-6, f"got {near_change_2}"
print("                      500 -> 250 m   250 -> 125 m")
print(f"well cell             {well_change_1:+10.3f} m   {well_change_2:+10.3f} m")
print(f"1 km west of the well {near_change_1:+10.3f} m   {near_change_2:+10.3f} m")

One kilometre from the well, the head changes by a few centimetres, and the
second change is about half the first. That is what a converging answer looks
like: each refinement changes it less than the one before. At 250 m this head
is within a few centimetres of where it is going.

The head in the well cell changes by more than a metre each time, and the
changes are not shrinking much. This is expected. Head near a well follows

$$h(r) = h_0 + \frac{Q}{2\pi T}\ln\frac{r}{r_0}$$

which keeps falling as $r$ gets smaller. MODFLOW reports one head per cell, and
that head represents the whole cell. A smaller cell represents a region closer
to the well, so its head is lower. With $Q$ = 20,000 m³/d and $T$ = 2,400 m²/d,
each halving lowers the well-cell head by roughly
$\frac{Q}{2\pi T}\ln 2 \approx 0.9$ m, however small the cells already are.

Two things follow:

- The head in a pumping cell is not the water level in the well. The water
  level in the well is lower. To estimate it you need an analytical correction
  or a much finer grid around the well.
- To decide whether a grid is fine enough, look at heads away from the wells
  and at the water budget. Those do settle down.

In [ ]:
sizes = sorted(results, reverse=True)
n_cells = [results[s]["n_cells"] for s in sizes]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
axes[0].plot(n_cells, [results[s]["well_head"] for s in sizes], "o-", color="#AB0520")
axes[0].set_title("Head in the well cell")
axes[1].plot(n_cells, [results[s]["near_head"] for s in sizes], "s-", color="#0C234B")
axes[1].set_title("Head 1 km west of the well")
for ax in axes:
    ax.set_xscale("log")
    ax.set_xlabel("number of cells")
    ax.set_ylabel("head (m)")
    ax.grid(alpha=0.3)
span = np.ptp(axes[0].get_ylim())                  # same vertical span on both panels
mid = results[250.0]["near_head"]
axes[1].set_ylim(mid - span / 2, mid + span / 2)
plt.tight_layout()
plt.show()

---

## Part 4: `ibound` and the shape of the aquifer

`ibound` is an array with one integer per cell. You set it in the BAS package,
as you did in Week 7.

| Value | Meaning | Head |
|---|---|---|
| `1` | active | calculated by MODFLOW |
| `0` | inactive | not calculated; the cell is not part of the model |
| `-1` | constant head | fixed at the value in `strt` |

The grid is always a rectangle. The aquifer usually is not, and `ibound = 0` is
how you leave cells out.

Here the Santa Catalina Mountains are bedrock. We do not have a map of where
the basin fill ends, so we use a simple rule in its place: any cell with land
surface above 850 m is bedrock and is inactive.

The west edge is where the basin continues toward the Santa Cruz River, so it
gets a constant head of 620 m. The east edge is mostly mountain front, so it is
left as a no-flow boundary.

In [ ]:
BEDROCK_CUTOFF = 850.0

ibound = np.ones((1, NROW, NCOL), dtype=int)
ibound[0][land_surface > BEDROCK_CUTOFF] = 0     # bedrock
ibound[:, :, 0] = -1                             # constant head on the west edge

print(f"{int((ibound == 1).sum())} active, {int((ibound == 0).sum())} inactive, "
      f"{int((ibound == -1).sum())} constant head")


def plot_ibound(ibound, title):
    fig, ax = plt.subplots(figsize=(8, 4.6))
    cmap = plt.matplotlib.colors.ListedColormap(["#AB0520", "#E2E9EB", "#1E5288"])
    im = ax.imshow(ibound[0], cmap=cmap, vmin=-1.5, vmax=1.5, extent=EXTENT)
    cb = fig.colorbar(im, ax=ax, ticks=[-1, 0, 1], shrink=0.85)
    cb.ax.set_yticklabels(["-1 constant head", "0 inactive", "1 active"])
    ax.set_xlabel("distance east (km)")
    ax.set_ylabel("distance north (km)")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


plot_ibound(ibound, "ibound with bedrock above 850 m inactive")

> **`ibound = 0` and `hk = 0` are different.** An inactive cell is left out of
> the equations. A cell with `hk = 0` is still in the equations and still gets
> a head, and cells like that can slow the solver down or stop it converging.
> To leave rock out of the model, use `ibound = 0`.

Now run the model with this `ibound`. Everything else is the same as Part 3.

In [ ]:
def basin_model(ibound, name):
    """Run the 250 m model with a given ibound. Returns (success, head)."""
    ws = RUN / f"week08_{name}"
    ws.mkdir(parents=True, exist_ok=True)

    mf = flopy.modflow.Modflow("basin", model_ws=str(ws), exe_name=str(MF_EXE))
    flopy.modflow.ModflowDis(mf, 1, NROW, NCOL, delr=DELR, delc=DELR,
                             top=600.0, botm=400.0, nper=1, steady=True)
    strt = np.full((1, NROW, NCOL), 620.0)
    flopy.modflow.ModflowBas(mf, ibound=ibound, strt=strt)
    flopy.modflow.ModflowLpf(mf, hk=12.0, laytyp=0, ipakcb=53)
    flopy.modflow.ModflowWel(mf, stress_period_data={0: [[0, 20, 30, -20000.0]]},
                             ipakcb=53)
    flopy.modflow.ModflowRch(mf, rech=1.2e-4, ipakcb=53)
    flopy.modflow.ModflowPcg(mf)
    flopy.modflow.ModflowOc(mf, stress_period_data={(0, 0): ["save head", "save budget"]})

    mf.write_input()
    ok, buff = mf.run_model(silent=True, report=True)
    head = flopy.utils.HeadFile(str(ws / "basin.hds")).get_data()
    return ok, head


ok_first, head_first = basin_model(ibound, "basin_first")
print(f"success = {ok_first}")
print(f"highest head in an active cell: {head_first[0][ibound[0] == 1].max():,.0f} m")

The run fails, and the highest head is far above anything physical: hundreds
of kilometres above the land surface.

The cause is in the `ibound` map. Along the east edge there are a few small
patches of active cells that sit below 850 m but are surrounded by inactive
cells. They receive recharge and have no way to pass it on, so no steady head
exists for them.

MODFLOW only moves water between cells that share a face. Two active cells that
touch at a corner are not connected.

`scipy.ndimage.label` finds groups of connected cells. It numbers each group 1,
2, 3, ... and puts 0 everywhere else. By default it uses the same rule as
MODFLOW: shared faces only.

In [ ]:
in_model = ibound[0] != 0                      # active or constant head
regions, n_regions = ndimage.label(in_model)
print(f"{n_regions} separate regions")

### YOUR TURN 2

Find out how big each region is and how many cells are outside the main one.

- `cells_per_region`: an array with the number of cells in region 1, region 2,
  and so on (length `n_regions`)
- `main_region`: the number of the largest region
- `n_cut_off`: how many cells are in the model but not in the main region

`regions == k` is `True` for the cells in region `k`.

In [ ]:
# YOUR TURN
cells_per_region = ...
main_region = ...
n_cut_off = ...

In [ ]:
# CHECK
cells_per_region = np.asarray(cells_per_region)
assert cells_per_region.shape == (n_regions,), f"expected {n_regions} values"
assert cells_per_region.sum() == in_model.sum(), "the counts should add up to every cell in the model"
assert cells_per_region[main_region - 1] == cells_per_region.max(), "main_region should be the largest"
assert n_cut_off == in_model.sum() - cells_per_region.max(), f"got {n_cut_off}"
print(f"cells per region: {cells_per_region.tolist()}")
print(f"main region: {main_region}")
print(f"cells cut off from it: {n_cut_off}")

### YOUR TURN 3

Remove the cut-off cells from the model and run it again.

- `ibound_fixed`: a copy of `ibound` with every cell outside the main region
  set to 0. Cells that were already 0 stay 0.
- `ok_fixed`, `head_fixed`: the result of `basin_model(ibound_fixed, "basin_fixed")`

In [ ]:
# YOUR TURN
ibound_fixed = ...
ok_fixed, head_fixed = ...

In [ ]:
# CHECK
assert ibound_fixed.shape == ibound.shape, f"got shape {ibound_fixed.shape}"
assert ndimage.label(ibound_fixed[0] != 0)[1] == 1, "there is still more than one region"
assert (ibound_fixed == 0).sum() == (ibound == 0).sum() + n_cut_off, (
    "only the cut-off cells should have changed"
)
assert (ibound_fixed[:, :, 0] == -1).all(), "the west edge should still be constant head"
assert ok_fixed, "the model should converge now"
active_heads = head_fixed[0][ibound_fixed[0] != 0]
print(f"success = {ok_fixed}")
print(f"head {active_heads.min():.1f} to {active_heads.max():.1f} m")

The model now runs. Removing those cells changes the model very little, since
they were never connected to the aquifer. Record the change anyway, because
anyone who rebuilds the model needs to make the same one.

MODFLOW did not report the cut-off cells as a problem. It reported a solver
failure, and the cause had to be found by looking at `ibound`. If there had
been no recharge on those cells, the run would have succeeded and the cells
would have kept their starting head. Checking that every active cell connects
to the rest of the model is something to do each time you build `ibound`.

In [ ]:
head_plot = np.where(ibound_fixed[0] == 0, np.nan, head_fixed[0])

fig, ax = plt.subplots(figsize=(8, 4.6))
im = ax.imshow(head_plot, cmap="Blues_r", extent=EXTENT)
fig.colorbar(im, ax=ax, label="head (m)", shrink=0.85)
ax.contour(np.flipud(head_plot), levels=8, colors="k", linewidths=0.6, extent=EXTENT)
ax.set_xlabel("distance east (km)")
ax.set_ylabel("distance north (km)")
ax.set_title("Head in the basin-shaped model (inactive cells blank)")
plt.tight_layout()
plt.show()

Inactive cells are blank because we set them to `nan` before plotting. In the
head file they hold a placeholder value, `hnoflo`, which is `-999.99` unless you
change it. Always mask inactive cells before you plot heads or take a minimum.

---

## Before you leave

1. *Kernel → Restart Kernel and Run All Cells*
2. Fix anything that breaks
3. Save

## Thursday

`week08_02_unconfined_dry_cells.ipynb`: the same aquifer as a water-table
aquifer, a model that does not converge with the default solver settings, and
dry cells.

## Stuck?

- This notebook runs five MODFLOW models. Each takes a second or two.
- An `AssertionError` from `confined_box` means the cell size does not divide
  the domain evenly. Use 500, 250, or 125.
- `basin_model` returning `success = False` before YOUR TURN 3 is expected.
- Office hours: Tuesdays 1:00–2:00pm, Harshbarger 322B.